# **Preparation Notebook**



---
## Setup Environment

In [79]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT1",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.11 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.41 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).

You can now save your data files in: /content/gdrive/MyDrive/36106/assignment/AT1/data


---
## Student Information

In [80]:
student_name = "SUSHRUTA GANGADHAR PATIL"
student_id = "26273312"

In [81]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_name', value=student_name)

In [82]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [83]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
!pip install "vegafusion"
!pip install "vl-convert-python>=1.6.0"

### 0.b Import Packages

In [84]:
# DO NOT MODIFY THE CODE IN THIS CELL
import pandas as pd
import altair as alt

In [85]:
alt.data_transformers.enable("vegafusion")
from sklearn.preprocessing import StandardScaler

---
## A. Feature Selection


In [86]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Load training data
try:
  training_df = pd.read_csv(at.folder_path / "vehicles_price_training.csv")
  validation_df = pd.read_csv(at.folder_path / "vehicles_price_validation.csv")
  testing_df = pd.read_csv(at.folder_path / "vehicles_price_testing.csv")
except Exception as e:
  print(e)

### A.1 Approach 1: "Distribution and Data Quality Analysis"

In [87]:
feature_selection_1_insights = """

The first thing I looked at in EDA was how the data was distributed and the quality of the data itself
as the outliers greatly affect the model.

For missing values, seats had 1,103 missing values and doors had 1,036 missing values. Also data
issues such as seats in doors column along with missing values made me to not consider them for
model training.
fuel_consumption had 1,103 missing values. There were also strings like 9.2 L / 100 km where the
numeric part needed to be extracted. It would have been a very important factor in determining the
price but the coorelation was weak, engine_cyclinders and brand_body_type, it would seem redundant
and hence will be dropped.
vehicle_condition was almost entirely USED with only 5 records saying NEW out of 8,774 on
training data. Test data consists majorly of NEW and DEMO models where the model will struggle as
the training data has very less to no values of such category.
vehicle_type had 307 unique values but when I looked they were clearly dealership names like Western
Highway Honda and Cars Connect. SO, this column is unreliable and will be dropped.
model_name had 595 unique values but wouldn't make much of an impact and brand_body type would cover
it, so will not be considered.

For the columns we kept, missing values were manageable:
1. body_type: 197 missing (2.2%) - will impute with mode
2. transmission_type: 0 missing but 101 records with dash placeholder
3. engine_cylinders: 0 missing but 1,080 records with dash and 5 records with 0 L, which are electric.

For the target variable "price", after parsing to float 7 values came up as missing and they will be
dropped as we cant impute the target variable.

Other suspicions that will be handled in data cleaning:
1. Alfa Romeo Sedan was priced at $88 which is clearly a data entry error
2. 3 records with under 100 km on used cars
3. A Nissan Juke from year 2000 with only 6 km listed as NEW, either the year or km is wrong

"""

In [88]:
_# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_1_insights', value=feature_selection_1_insights)

### A.2 Approach 2: "Relationship with Target Variable"

In [89]:
feature_selection_2_insights = """

My second approach was to look at the features relationship with the target variable. For this I
used correlation for numeric attributes and box plots for categorical variables.

For numeric features:
kilometres_driven had the strongest correlation with price at -0.27. The scatter plot showed a clear
cone shape where low km cars have a wide price range but high km cars are almost all cheap.

For categorical features:
body_type showed the clearest price differences. Coupes had a median price of $33,500 while Hatchbacks
sat at $14,990, over $18,000 difference. The individual boxplots showed distinct price distributions
for each body type with Convertibles also commanding a premium and having the widest spread.

vehicle_brand showed massive price differences from Daewoo at the bottom to Lamborghini at the top.
The split boxplots for top 33 and bottom 33 brands showed that expensive brands like Ferrari and
McLaren have wide price ranges driven by specific models, while mainstream brands like Toyota and
Holden have tight predictable ranges. However brand alone is misleading, a Toyota SUV and Toyota
Hatchback are priced completely differently. This led to the decision to engineer brand_bodytype as
a combined feature.

engine_cylinders showed a very consistent pattern, 12 cyl had amedian over $120,000 while 3 cyl sat
at the bottom. One of the strongest categorical signals we found.

drive_type showed 4WD and AWD commanding significantly higher prices than Front wheel drive at around
$25,000 vs $16,000 median. This reflects the premium for SUVs and utes in the Australian market.

transmission_type showed Automatic cars priced around $3,000 higher than Manual on average, consistent
with Australian market preference.

vehicle_condition showed almost no price variation since 99.9% of records are USED in the training data.
So, it was dropped.

fuel_consumption had a correlation of only +0.08 with price, the weakest of all numeric features and
with 12.6% missing. So it was dropped.

"""

In [90]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_2_insights', value=feature_selection_2_insights)

### A.3 Approach: "Business Use Case Alignment"

In [91]:
feature_selection_3_insights = """

Last check was more of a common sense one. This model is meant for someone browsing on Gumtree or
Facebook Marketplace, so whatever features I use have to actually appear on a normal listing. No point
using something a buyer would never see or know. kilometres_driven, manufacturing_year, body_type,
vehicle_brand, transmission_type, drive_type and engine_cylinders all are basic information
shared when listing.
The personal details like name, phone, email and address are irrelevant for the model and
also keeping that raises privacy issues. So they were dropped.
location seemed fine at first as location is also a factor of the price but then the test set
had 449 missing values and location can no way be easily imputed and wrong value could mean wrong
prediction which was a problem. So, it was dropped.
vehicle_colour is one a buyer could technically enter but I believe colour doesnt really affect
used car prices much, at least not in this price range. So, it will not be considered.

model_name was tempting but 595 unique values is just too many to encode in any reliable way.
And when there is brand and body type together, it will capture most of what model name would add anyway.

"""

In [92]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_3_insights', value=feature_selection_3_insights)

### A.z Final Selection of Features

In [93]:
features_list = [
    "kilometres_driven",
    "manufacturing_year",
    "body_type",
    "vehicle_brand",
    "transmission_type",
    "drive_type",
    "engine_cylinders",
    "price"
]

target_name = "price"

In [94]:
feature_selection_explanations = """

After analysing the data from three different angles in the EDA
notebook - data quality, relationship with price, and business use
case alignment - all three approaches pointed to the same set of
7 features.

Features selected and why:
1. kilometres_driven - the strongest numeric predictor with a
   correlation of -0.27 with price. More km means more wear which
   means lower price. A buyer would always know this from a listing.
2. manufacturing_year - weak correlation alone at +0.10 but still
   carries useful information.
3. body_type - one of the strongest categorical features. The price
   difference between a Coupe ($33,500 median) and a Hatchback
   ($14,990 median) is over $18,000. Clear and consistent signal.
4. vehicle_brand - massive price differences across brands from
   Daewoo at the bottom to Lamborghini at the top. Will be combined
   with body_type in feature engineering to create an even stronger
   signal.
5. transmission_type - Automatic cars command a premium over Manual
   in the Australian market. Clear and consistent relationship.
6. drive_type - 4WD and AWD vehicles are priced significantly higher
   than Front wheel drive, reflecting the SUV and ute premium.
7. engine_cylinders - very strong signal. More cylinders consistently
   means higher price, from 3 cyl at the bottom to 12 cyl at the top.

"""

In [95]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## B. Data Cleaning

In [96]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets
try:
  training_df_clean = training_df[features_list].copy()
  validation_df_clean = validation_df[features_list].copy()
  testing_df_clean = testing_df[features_list].copy()
except Exception as e:
  print(e)

### B.1 Fixing "Price"

In [97]:
# parse price to float for all three splits
training_df_clean["price"] = pd.to_numeric(
    training_df_clean["price"].astype(str).str.replace(",", ""), errors="coerce"
)
validation_df_clean["price"] = pd.to_numeric(
    validation_df_clean["price"].astype(str).str.replace(",", ""), errors="coerce"
)
testing_df_clean["price"] = pd.to_numeric(
    testing_df_clean["price"].astype(str).str.replace(",", ""), errors="coerce"
)

print("price dtype:", training_df_clean["price"].dtype)
print(f"Missing price values in training: {training_df_clean['price'].isna().sum()}")
print(f"Missing price values in validation: {validation_df_clean['price'].isna().sum()}")
print(f"Missing price values in testing: {testing_df_clean['price'].isna().sum()}")

price dtype: float64
Missing price values in training: 7
Missing price values in validation: 1
Missing price values in testing: 41


In [98]:
# checking what values failed to parse in testing
test_price_raw = testing_df_clean["price"].astype(str)
failed_test = testing_df_clean[pd.to_numeric(test_price_raw.str.replace(",", ""), errors="coerce").isna()]["price"]
print(f"Failed to parse in testing during EDA: {len(failed_test)}")
print(failed_test.values)

Failed to parse in testing during EDA: 41
[nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan]


In [99]:
# dropping missing price rows across all splits

training_df_clean = training_df_clean.dropna(subset=["price"])
validation_df_clean = validation_df_clean.dropna(subset=["price"])
testing_df_clean = testing_df_clean.dropna(subset=["price"])

print(f"Training : {len(training_df_clean):,} rows remaining")
print(f"Validation: {len(validation_df_clean):,} rows remaining")
print(f"Testing  : {len(testing_df_clean):,} rows remaining")

Training : 8,767 rows remaining
Validation: 2,614 rows remaining
Testing  : 2,606 rows remaining


In [100]:
# dropping the $88 Alfa Romeo record identified in EDA as a clear data entry error
training_df_clean = training_df_clean[training_df_clean["price"] != 88]
print(f"Training: {len(training_df_clean):,} rows remaining")

Training: 8,766 rows remaining


In [101]:
data_cleaning_1_explanations = """

Price came in as a string in the raw CSV so got it converted to float. There was 41 missing in test,
1 in validation and 7 in training.
All rows with a missing price were dropped
$88 Alfa Romeo Sedan from training, a 2006 model with 90,000 km that was identified in EDA as a clear
data entry error was also dropped.
The $900 Daewoo Lanos was reviewed and kept since a 1998 car with 310,000 km selling for $900 is actually
plausible.

Final row counts after cleaning:
Training  : 8,766 rows
Validation: 2,614 rows
Testing   : 2,606 rows

"""

In [102]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_1_explanations', value=data_cleaning_1_explanations)

### B.2 Fixing "kilometers_driven"

In [103]:
# parsing kilometres_driven from string to float across all splits
training_df_clean["kilometres_driven"] = pd.to_numeric(
    training_df_clean["kilometres_driven"].astype(str).str.replace(",", ""), errors="coerce"
)
validation_df_clean["kilometres_driven"] = pd.to_numeric(
    validation_df_clean["kilometres_driven"].astype(str).str.replace(",", ""), errors="coerce"
)
testing_df_clean["kilometres_driven"] = pd.to_numeric(
    testing_df_clean["kilometres_driven"].astype(str).str.replace(",", ""), errors="coerce"
)

print("kilometres_driven dtype:", training_df_clean["kilometres_driven"].dtype)
print(f"Missing - training  : {training_df_clean['kilometres_driven'].isna().sum()}")
print(f"Missing - validation: {validation_df_clean['kilometres_driven'].isna().sum()}")
print(f"Missing - testing   : {testing_df_clean['kilometres_driven'].isna().sum()}")

print(f"Training  : {len(training_df_clean):,} rows remaining")
print(f"Validation: {len(validation_df_clean):,} rows remaining")
print(f"Testing   : {len(testing_df_clean):,} rows remaining")

kilometres_driven dtype: float64
Missing - training  : 11
Missing - validation: 0
Missing - testing   : 564
Training  : 8,766 rows remaining
Validation: 2,614 rows remaining
Testing   : 2,606 rows remaining


In [104]:
# checking what the raw km values look like in testing
failed_test_km = testing_df_clean[testing_df_clean["kilometres_driven"].isna()]
print(f"Rows with missing km in testing: {len(failed_test_km)}")
print()
print("Raw km values at those indices:")
print(testing_df.loc[failed_test_km.index, "kilometres_driven"].value_counts().head(10))

Rows with missing km in testing: 564

Raw km values at those indices:
kilometres_driven
- / -    449
-        115
Name: count, dtype: int64


In [105]:
# dropping rows with missing km across all splits
# km cannot be reliably imputed - unknown km could be anything from nearly new to very high mileage

training_df_clean = training_df_clean.dropna(subset=["kilometres_driven"])
validation_df_clean = validation_df_clean.dropna(subset=["kilometres_driven"])
testing_df_clean = testing_df_clean.dropna(subset=["kilometres_driven"])

print(f"Training  : {len(training_df_clean):,} rows remaining")
print(f"Validation: {len(validation_df_clean):,} rows remaining")
print(f"Testing   : {len(testing_df_clean):,} rows remaining")

Training  : 8,755 rows remaining
Validation: 2,614 rows remaining
Testing   : 2,042 rows remaining


In [106]:
data_cleaning_2_explanations = """

Same as price, kilometres_driven was stored as string with missing and invalid values, so it was
converted to float.
After parsing, missing values were found across all three splits:
Training  : 11 missing
Validation: 0 missing
Testing   : 564 missing

The 564 missing values in testing, they were not actual nulls but placeholders like - and - / -
which isnull().sum() in EDA could not detect since they are valid strings, not null values.

Unlike body_type or transmission_type where imputing with the mode is reasonable, kilometres_driven
cannot be imputed. A car with unknown km could be anything from nearly new to very high
mileage. Filling it with the median would mislead the model.

For this reason all rows with missing km were dropped across all three splits. The 11 dropped from
training is negligible. The 564 dropped from testing is larger but since test data is only used for
final evaluation it does not affect model training.

Final row counts after cleaning:
Training  : 8,754 rows
Validation: 2,614 rows
Testing   : 2,042 rows

"""

In [107]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_2_explanations', value=data_cleaning_2_explanations)

### B.3 Fixing "body_type"

In [108]:
# checking missing body_type before imputation
print(f"Missing body_type - training  : {training_df_clean['body_type'].isna().sum()}")
print(f"Missing body_type - validation: {validation_df_clean['body_type'].isna().sum()}")
print(f"Missing body_type - testing   : {testing_df_clean['body_type'].isna().sum()}")

# fit mode on training only - never use val or test to calculate this
body_mode = training_df_clean["body_type"].mode()[0]
print(f"\nMode from training: {body_mode}")

# apply same mode to all three splits
training_df_clean["body_type"] = training_df_clean["body_type"].fillna(body_mode)
validation_df_clean["body_type"] = validation_df_clean["body_type"].fillna(body_mode)
testing_df_clean["body_type"] = testing_df_clean["body_type"].fillna(body_mode)

print(f"\nAfter imputation:")
print(f"Training  : {training_df_clean['body_type'].isna().sum()}")
print(f"Validation: {validation_df_clean['body_type'].isna().sum()}")
print(f"Testing   : {testing_df_clean['body_type'].isna().sum()}")

Missing body_type - training  : 194
Missing body_type - validation: 30
Missing body_type - testing   : 12

Mode from training: SUV

After imputation:
Training  : 0
Validation: 0
Testing   : 0


In [109]:
print("Training data:\n", training_df_clean["body_type"].value_counts())
print("\nValidation data:\n", validation_df_clean["body_type"].value_counts())
print("\nTesting data:\n", testing_df_clean["body_type"].value_counts())

Training data:
 body_type
SUV             3104
Hatchback       1492
Ute / Tray      1348
Sedan           1309
Wagon            804
Commercial       363
Coupe            221
Convertible       99
People Mover      15
Name: count, dtype: int64

Validation data:
 body_type
SUV             1280
Hatchback        467
Ute / Tray       345
Sedan            271
Wagon            135
Commercial        71
Coupe             31
Convertible        7
Other              6
People Mover       1
Name: count, dtype: int64

Testing data:
 body_type
SUV             1238
Ute / Tray       284
Hatchback        218
Wagon            125
Sedan            119
Commercial        30
Coupe             21
Other              4
Convertible        2
People Mover       1
Name: count, dtype: int64


In [110]:
# Drop unseen 'Other' body_type records — not present in training,
before_val = len(validation_df_clean)
before_test = len(testing_df_clean)

validation_df_clean = validation_df_clean[validation_df_clean["body_type"] != "Other"]
testing_df_clean = testing_df_clean[testing_df_clean["body_type"] != "Other"]

print(f"Validation: dropped {before_val - len(validation_df_clean)} 'Other' body_type rows — {len(validation_df_clean):,} remaining")
print(f"Testing   : dropped {before_test - len(testing_df_clean)} 'Other' body_type rows — {len(testing_df_clean):,} remaining")

Validation: dropped 6 'Other' body_type rows — 2,608 remaining
Testing   : dropped 4 'Other' body_type rows — 2,038 remaining


In [111]:
data_cleaning_3_explanations = """

When I looked at the body_type column I found missing values across all three splits:
Training  : 194 missing
Validation: 30 missing
Testing   : 12 missing

Since body_type is one of our strongest features with clear price differences across categories I
did not want to drop these rows and lose useful data. Instead I imputed them with the most frequent
value (mode) from the training set which was SUV.

The mode was calculated from training only and the same value was applied to validation and test to
avoid data leakage.The number of missing entries is also relatively small compared to the total
dataset so imputing with mode does not significantly affect the distribution.

One important observation I noticed is that validation and test sets contain an Other category in
body_type which is not present in training data, 6 records in validation and 4 in
testing. This is an unseen category that the model will not know
how to handle during encoding. So, I decided on dropping it

"""

In [112]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_3_explanations', value=data_cleaning_3_explanations)

### B.4 Fixing "vehicle_brand"

In [113]:
print(f"Missing vehicle_brand - training  : {training_df_clean['vehicle_brand'].isna().sum()}")
print(f"Missing vehicle_brand - validation: {validation_df_clean['vehicle_brand'].isna().sum()}")
print(f"Missing vehicle_brand - testing   : {testing_df_clean['vehicle_brand'].isna().sum()}")

print(f"\nUnique brands (training): {training_df_clean['vehicle_brand'].nunique()}")
print(f"\nUnique brands (training): {validation_df_clean['vehicle_brand'].nunique()}")
print(f"\nUnique brands (training): {testing_df_clean['vehicle_brand'].nunique()}")

Missing vehicle_brand - training  : 0
Missing vehicle_brand - validation: 0
Missing vehicle_brand - testing   : 0

Unique brands (training): 66

Unique brands (training): 46

Unique brands (training): 43


In [114]:
# unique sets
train_brands = set(training_df_clean["vehicle_brand"].unique())
val_brands = set(validation_df_clean["vehicle_brand"].unique())
test_brands = set(testing_df_clean["vehicle_brand"].unique())

# differences
val_not_in_train = val_brands - train_brands
test_not_in_train = test_brands - train_brands

print(f"Brands in val but NOT in training: {len(val_not_in_train)}")
print(val_not_in_train)

print(f"\nBrands in test but NOT in training: {len(test_not_in_train)}")
print(test_not_in_train)

Brands in val but NOT in training: 1
{'Genesis'}

Brands in test but NOT in training: 4
{'Genesis', 'Abarth', 'BYD', 'Cupra'}


In [115]:
data_cleaning_4_explanations = """

vehicle_brand had no missing values across all three splits. But when I compared the unique brands,
I noticed that validation and test have brands that never appeared in training:
Validation: Genesis
Test: Genesis, Cupra, Abarth, BYD

Training covers vehicles up to 2017 and validation and test cover 2019-2023. Brands like BYD
and Cupra were not available back then. So the model has never seen these brands before.

This becomes a problem when we do target encoding in Section C. We calculate mean prices per brand
from training, but these new brands have no training data so there is nothing to map them to.

So, the plan is to map any unseen brand to the global mean price from training during target encoding.
It is the safest solution as we do not know how these brands are priced.

"""

In [116]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_4_explanations', value=data_cleaning_4_explanations)

### B.5 Fixing "transmission_type"

In [117]:
# checking transmission_type
print("transmission_type values:")
print(training_df_clean["transmission_type"].value_counts())
print(f"\nDash placeholders - training  : {(training_df_clean['transmission_type'] == '-').sum()}")
print(f"Dash placeholders - validation: {(validation_df_clean['transmission_type'] == '-').sum()}")
print(f"Dash placeholders - testing   : {(testing_df_clean['transmission_type'] == '-').sum()}")

transmission_type values:
transmission_type
Automatic    7112
Manual       1546
-              97
Name: count, dtype: int64

Dash placeholders - training  : 97
Dash placeholders - validation: 25
Dash placeholders - testing   : 56


In [118]:
# replace dash with mode from training
trans_mode = training_df_clean[training_df_clean["transmission_type"] != "-"]["transmission_type"].mode()[0]
print(f"Mode from training (excluding dash): {trans_mode}")

training_df_clean["transmission_type"] = training_df_clean["transmission_type"].replace("-", trans_mode)
validation_df_clean["transmission_type"] = validation_df_clean["transmission_type"].replace("-", trans_mode)
testing_df_clean["transmission_type"] = testing_df_clean["transmission_type"].replace("-", trans_mode)

Mode from training (excluding dash): Automatic


In [119]:
print("transmission_type - training:")
print(training_df_clean["transmission_type"].value_counts())
print()
print("transmission_type - validation:")
print(validation_df_clean["transmission_type"].value_counts())
print()
print("transmission_type - testing:")
print(testing_df_clean["transmission_type"].value_counts())

transmission_type - training:
transmission_type
Automatic    7209
Manual       1546
Name: count, dtype: int64

transmission_type - validation:
transmission_type
Automatic    2496
Manual        112
Name: count, dtype: int64

transmission_type - testing:
transmission_type
Automatic    1956
Manual         82
Name: count, dtype: int64


In [120]:
data_cleaning_5_explanations = """

transmission_type had 97 records in training, 25 in validation and 60 in testing with dash as the value.
They were replaced with the mode from training which was Automatic.
The same mode value was applied to all three splits using the value calculated from training to avoid
data leakage.

"""

In [121]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_4_explanations', value=data_cleaning_4_explanations)

### B.6 Fixing "engine_cylinders"

In [122]:
# checking engine_cylinders
print("engine_cylinders values:")
print(training_df_clean["engine_cylinders"].value_counts())


engine_cylinders values:
engine_cylinders
4 cyl     5557
6 cyl     1409
-         1071
8 cyl      384
5 cyl      265
3 cyl       48
12 cyl      10
0 L          5
2 cyl        4
10 cyl       2
Name: count, dtype: int64


In [123]:
print(f"\nDash placeholders - training  : {(training_df_clean['engine_cylinders'] == '-').sum()}")
print(f"Dash placeholders - validation: {(validation_df_clean['engine_cylinders'] == '-').sum()}")
print(f"Dash placeholders - testing   : {(testing_df_clean['engine_cylinders'] == '-').sum()}")
print(f"\n0 L values - training  : {(training_df_clean['engine_cylinders'] == '0 L').sum()}")
print(f"0 L values - validation: {(validation_df_clean['engine_cylinders'] == '0 L').sum()}")
print(f"0 L values - testing   : {(testing_df_clean['engine_cylinders'] == '0 L').sum()}")


Dash placeholders - training  : 1071
Dash placeholders - validation: 143
Dash placeholders - testing   : 258

0 L values - training  : 5
0 L values - validation: 13
0 L values - testing   : 48


In [124]:
# Replace only dash with mode — genuinely missing values
cyl_mode = training_df_clean[
    ~training_df_clean["engine_cylinders"].isin(["-", "0 L"])
]["engine_cylinders"].mode()[0]
print(f"Mode from training (excluding dash and 0 L): {cyl_mode}")

training_df_clean["engine_cylinders"] = training_df_clean["engine_cylinders"].replace({"-": cyl_mode})
validation_df_clean["engine_cylinders"] = validation_df_clean["engine_cylinders"].replace({"-": cyl_mode})
testing_df_clean["engine_cylinders"] = testing_df_clean["engine_cylinders"].replace({"-": cyl_mode})

# 0 L = Electric vehicles — kept as 0 cylinders, this is correct
training_df_clean["engine_cylinders"] = training_df_clean["engine_cylinders"].replace({"0 L": "0"})
validation_df_clean["engine_cylinders"] = validation_df_clean["engine_cylinders"].replace({"0 L": "0"})
testing_df_clean["engine_cylinders"] = testing_df_clean["engine_cylinders"].replace({"0 L": "0"})

print(f"\nAfter replacing:")
print(training_df_clean["engine_cylinders"].value_counts())
print(f"\nMissing - training  : {training_df_clean['engine_cylinders'].isna().sum()}")
print(f"Missing - validation: {validation_df_clean['engine_cylinders'].isna().sum()}")
print(f"Missing - testing   : {testing_df_clean['engine_cylinders'].isna().sum()}")

Mode from training (excluding dash and 0 L): 4 cyl

After replacing:
engine_cylinders
4 cyl     6628
6 cyl     1409
8 cyl      384
5 cyl      265
3 cyl       48
12 cyl      10
0            5
2 cyl        4
10 cyl       2
Name: count, dtype: int64

Missing - training  : 0
Missing - validation: 0
Missing - testing   : 0


In [125]:
# extract numeric part from engine_cylinders
# e.g. "4 cyl" -> 4, "6 cyl" -> 6

training_df_clean["engine_cylinders"] = training_df_clean["engine_cylinders"].str.extract(r"(\d+)").astype(int)
validation_df_clean["engine_cylinders"] = validation_df_clean["engine_cylinders"].str.extract(r"(\d+)").astype(int)
testing_df_clean["engine_cylinders"] = testing_df_clean["engine_cylinders"].str.extract(r"(\d+)").astype(int)

print(training_df_clean["engine_cylinders"].value_counts())

engine_cylinders
4     6628
6     1409
8      384
5      265
3       48
12      10
0        5
2        4
10       2
Name: count, dtype: int64


In [126]:
data_cleaning_6_explanations = """

engine_cylinders had two different placeholder problems; dashes and 0 L values.

The dashes were straightforward missing values. 1,071 in training, 143 in validation, 258 in testing.
These were replaced with the mode from training which was 4 cyl. With 63% of training data being 4 cyl,
assuming an unknown is most likely a 4 cyl is a pretty safe bet.

The 0 L were electric cars so they were genuine data(Tesla Model S, BMW i3, Nissan Leaf). So, these
were kept as 0 cylinders rather than replaced with the mode.

After handling the placeholders the numeric part was extracted from the string as it is an integer
for the model to use directly.

"""

In [127]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_6_explanations', value=data_cleaning_6_explanations)

### B.7 Fixing "manufacturing_year"

In [128]:
# manufacturing_year is stored as float64 but should be int

print(f"Before - dtype: {training_df_clean['manufacturing_year'].dtype}")
print(f"Sample: {training_df_clean['manufacturing_year'].head(5).values}")

training_df_clean["manufacturing_year"] = training_df_clean["manufacturing_year"].astype(int)
validation_df_clean["manufacturing_year"] = validation_df_clean["manufacturing_year"].astype(int)
testing_df_clean["manufacturing_year"] = testing_df_clean["manufacturing_year"].astype(int)

print(f"\nAfter - dtype: {training_df_clean['manufacturing_year'].dtype}")
print(f"Sample: {training_df_clean['manufacturing_year'].head(5).values}")
print(f"\nRange - training  : {training_df_clean['manufacturing_year'].min()} - {training_df_clean['manufacturing_year'].max()}")
print(f"Range - validation: {validation_df_clean['manufacturing_year'].min()} - {validation_df_clean['manufacturing_year'].max()}")
print(f"Range - testing   : {testing_df_clean['manufacturing_year'].min()} - {testing_df_clean['manufacturing_year'].max()}")

Before - dtype: float64
Sample: [2014. 2013. 1978. 2006. 2012.]

After - dtype: int64
Sample: [2014 2013 1978 2006 2012]

Range - training  : 1940 - 2017
Range - validation: 2019 - 2020
Range - testing   : 2022 - 2023


In [129]:
data_cleaning_7_explanations = """

manufacturing_year was stored as float64 in the raw CSV showing values like 2015.0 which is not right.
So got it converted to int64.

No missing values found in this column so no imputation was needed.
After converting, the ranges confirmed the time based split we identified in EDA:
Training  : 1940 - 2017
Validation: 2019 - 2020
Testing   : 2022 - 2023

"""

In [130]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_7_explanations', value=data_cleaning_7_explanations)

---
## C. Feature Engineering

In [131]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets

try:
  training_df_eng = training_df_clean.copy()
  validation_df_eng = validation_df_clean.copy()
  testing_df_eng = testing_df_clean.copy()
except Exception as e:
  print(e)

### C.1 New Feature "brand_body_type"



In [132]:
training_df_eng["brand_bodytype"] = training_df_eng["vehicle_brand"] + "_" + training_df_eng["body_type"]
validation_df_eng["brand_bodytype"] = validation_df_eng["vehicle_brand"] + "_" + validation_df_eng["body_type"]
testing_df_eng["brand_bodytype"] = testing_df_eng["vehicle_brand"] + "_" + testing_df_eng["body_type"]

print(f"Unique brand_bodytype combinations - training  : {training_df_eng['brand_bodytype'].nunique()}")
print(f"Unique brand_bodytype combinations - validation: {validation_df_eng['brand_bodytype'].nunique()}")
print(f"Unique brand_bodytype combinations - testing   : {testing_df_eng['brand_bodytype'].nunique()}")
print()
print("Sample combinations:")
print(training_df_eng["brand_bodytype"].value_counts().head(10))

Unique brand_bodytype combinations - training  : 230
Unique brand_bodytype combinations - validation: 136
Unique brand_bodytype combinations - testing   : 131

Sample combinations:
brand_bodytype
Toyota_SUV           435
Nissan_SUV           297
Toyota_Ute / Tray    275
Holden_Sedan         262
Ford_Ute / Tray      245
Mitsubishi_SUV       241
Hyundai_Hatchback    230
Holden_Ute / Tray    224
Hyundai_SUV          208
Holden_SUV           204
Name: count, dtype: int64


In [133]:
# smoothed mean encoding for brand_bodytype
# formula: (n * combination_mean + k * global_mean) / (n + k)
k = 10

# calculating global mean from training
global_mean = training_df_eng["price"].mean()
print(f"Global mean price from training: ${global_mean:,.2f}")

combo_stats = training_df_eng.groupby("brand_bodytype")["price"].agg(["mean", "count"]).reset_index()
combo_stats.columns = ["brand_bodytype", "combo_mean", "combo_count"]

combo_stats["smoothed_mean"] = (
    (combo_stats["combo_count"] * combo_stats["combo_mean"]) + (k * global_mean)
) / (combo_stats["combo_count"] + k)

print(f"\nTop 10 most expensive combinations after smoothing:")
print(combo_stats.sort_values("smoothed_mean", ascending=False).head(10).to_string())
print(f"\nBottom 10 cheapest combinations after smoothing:")
print(combo_stats.sort_values("smoothed_mean").head(10).to_string())

Global mean price from training: $24,734.24

Top 10 most expensive combinations after smoothing:
          brand_bodytype     combo_mean  combo_count  smoothed_mean
43         Ferrari_Coupe  879940.000000            2  167268.536360
42   Ferrari_Convertible  277440.000000            4   96935.888309
111    Lamborghini_Coupe  649880.000000            1   81565.676029
176        Porsche_Coupe  219745.250000            4   80451.674023
65             HSV_Sedan   73375.709677           31   61511.937471
137  McLaren_Convertible  369880.000000            1   56111.130575
52            Ford_Coupe   67352.090909           22   54034.013635
187    Rolls-Royce_Sedan  329990.000000            1   52484.766938
66        HSV_Ute / Tray   98444.500000            6   52375.589770
175  Porsche_Convertible   81399.888889            9   51575.865070

Bottom 10 cheapest combinations after smoothing:
           brand_bodytype    combo_mean  combo_count  smoothed_mean
72       Holden_Hatchback  11533.6239

In [134]:
# mapping dictionary from training stats
encoding_map = dict(zip(combo_stats["brand_bodytype"], combo_stats["smoothed_mean"]))

training_df_eng["brand_bodytype_encoded"] = training_df_eng["brand_bodytype"].map(encoding_map).fillna(global_mean)
validation_df_eng["brand_bodytype_encoded"] = validation_df_eng["brand_bodytype"].map(encoding_map).fillna(global_mean)
testing_df_eng["brand_bodytype_encoded"] = testing_df_eng["brand_bodytype"].map(encoding_map).fillna(global_mean)

print(f"Missing after encoding - training  : {training_df_eng['brand_bodytype_encoded'].isna().sum()}")
print(f"Missing after encoding - validation: {validation_df_eng['brand_bodytype_encoded'].isna().sum()}")
print(f"Missing after encoding - testing   : {testing_df_eng['brand_bodytype_encoded'].isna().sum()}")
print()
print(f"Sample encoded values:")
print(training_df_eng[["brand_bodytype", "brand_bodytype_encoded"]].head(10))

Missing after encoding - training  : 0
Missing after encoding - validation: 0
Missing after encoding - testing   : 0

Sample encoded values:
          brand_bodytype  brand_bodytype_encoded
0        Honda_Hatchback            19980.992865
1        Ford_Ute / Tray            29292.001711
2    Porsche_Convertible            51575.865070
3            Honda_Sedan            16447.830221
4                Kia_SUV            22021.628636
5       Nissan_Hatchback            15263.687478
6             Toyota_SUV            31478.196486
7             Ford_Sedan            17310.792286
8  Mitsubishi_Ute / Tray            22457.938938
9      Nissan_Ute / Tray            22488.960606


In [135]:
# dropping the original brand_bodytype string column and vehicle_brand

training_df_eng = training_df_eng.drop(columns=["brand_bodytype", "vehicle_brand"])
validation_df_eng = validation_df_eng.drop(columns=["brand_bodytype", "vehicle_brand"])
testing_df_eng = testing_df_eng.drop(columns=["brand_bodytype", "vehicle_brand"])

print("Remaining columns:", training_df_eng.columns.tolist())

Remaining columns: ['kilometres_driven', 'manufacturing_year', 'body_type', 'transmission_type', 'drive_type', 'engine_cylinders', 'price', 'brand_bodytype_encoded']


In [136]:
feature_engineering_1_explanations = """

During EDA I discovered that vehicle_brand alone could be misleading as a feature, like a Toyota SUV
and a Toyota Hatchback are priced completely differently, so averaging all Toyotas together into one
number would lose important pricing information. This led to the idea of combining vehicle_brand and
body_type into a single feature called brand_bodytype.

The combination produced 220 unique combinations in training, ranging from Toyota_SUV to rare
combinations like Ferrari_Coupe and Lamborghini_Coupe.

For encoding I used smoothed mean encoding instead of regular target encoding. Regular target encoding
would simply replace each combination with its mean price from training. But rare combinations with
only 1-2 records would be encoded as exactly those records prices, which would cause severe overfitting.

Smoothed mean encoding fixes this by blending the combination mean with the global mean using this formula:
smoothed = (n * combo_mean + k * global_mean) / (n + k)

Where n is the number of records for that combination and k is the smoothing factor set to 10.

The encoding map was calculated from training and applied to all three splits. Unseen combinations
in validation and test that never appeared in training were mapped to the global mean price of
$24,737.

After encoding the original vehicle_brand and brand_bodytype string columns were dropped.

"""

In [137]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_1_explanations', value=feature_engineering_1_explanations)

### C.2 New Feature "body_drive_encoded"



In [138]:
# combining body_type and drive_type into one feature
# a SUV_4WD is genuinely priced differently from a SUV_Front

training_df_eng["body_drive"] = training_df_eng["body_type"] + "_" + training_df_eng["drive_type"]
validation_df_eng["body_drive"] = validation_df_eng["body_type"] + "_" + validation_df_eng["drive_type"]
testing_df_eng["body_drive"] = testing_df_eng["body_type"] + "_" + testing_df_eng["drive_type"]

print(f"Unique body_drive combinations - training  : {training_df_eng['body_drive'].nunique()}")
print(f"Unique body_drive combinations - validation: {validation_df_eng['body_drive'].nunique()}")
print(f"Unique body_drive combinations - testing   : {testing_df_eng['body_drive'].nunique()}")
print()
print("Sample combinations:")
print(training_df_eng["body_drive"].value_counts().head(10))

Unique body_drive combinations - training  : 37
Unique body_drive combinations - validation: 30
Unique body_drive combinations - testing   : 32

Sample combinations:
body_drive
Hatchback_Front    1311
SUV_AWD            1116
SUV_4WD             912
Ute / Tray_4WD      828
SUV_Front           787
Sedan_Front         678
Sedan_Rear          517
Ute / Tray_Rear     418
Wagon_Other         326
Wagon_Front         307
Name: count, dtype: int64


In [139]:
body_drive_stats = training_df_eng.groupby("body_drive")["price"].agg(["mean", "count"]).reset_index()
body_drive_stats.columns = ["body_drive", "combo_mean", "combo_count"]

body_drive_stats["smoothed_mean"] = (
    (body_drive_stats["combo_count"] * body_drive_stats["combo_mean"]) + (k * global_mean)
) / (body_drive_stats["combo_count"] + k)

print("Top 10 most expensive body_drive combinations:")
print(body_drive_stats.sort_values("smoothed_mean", ascending=False).head(10).to_string())
print()
print("Bottom 10 cheapest:")
print(body_drive_stats.sort_values("smoothed_mean").head(10).to_string())

Top 10 most expensive body_drive combinations:
          body_drive     combo_mean  combo_count  smoothed_mean
10         Coupe_AWD  102520.000000           15   71405.697453
8   Convertible_Rear   74381.254237           59   67186.035309
12       Coupe_Other   70825.181818           22   56421.763635
13        Coupe_Rear   57720.994048          168   55867.805822
5    Convertible_AWD   64119.571429            7   40951.731548
33         Wagon_AWD   34156.161290           31   31858.132593
27        Sedan_Rear   31871.926499          517   31736.486596
9          Coupe_4WD   86880.000000            1   30383.857847
30  Ute / Tray_Front   39722.166667            6   30354.714770
28    Ute / Tray_4WD   29690.922705          828   29631.773790

Bottom 10 cheapest:
           body_drive    combo_mean  combo_count  smoothed_mean
26        Sedan_Front  14359.920354          678   14510.709937
15    Hatchback_Front  15176.125095         1311   15248.480270
6   Convertible_Front  16918.892857 

In [140]:
body_drive_map = dict(zip(body_drive_stats["body_drive"], body_drive_stats["smoothed_mean"]))

training_df_eng["body_drive_encoded"] = training_df_eng["body_drive"].map(body_drive_map).fillna(global_mean)
validation_df_eng["body_drive_encoded"] = validation_df_eng["body_drive"].map(body_drive_map).fillna(global_mean)
testing_df_eng["body_drive_encoded"] = testing_df_eng["body_drive"].map(body_drive_map).fillna(global_mean)

# drop string columns no longer needed
training_df_eng = training_df_eng.drop(columns=["body_drive", "body_type", "drive_type"])
validation_df_eng = validation_df_eng.drop(columns=["body_drive", "body_type", "drive_type"])
testing_df_eng = testing_df_eng.drop(columns=["body_drive", "body_type", "drive_type"])

print(f"\nMissing after encoding - training  : {training_df_eng['body_drive_encoded'].isna().sum()}")
print(f"Missing after encoding - validation: {validation_df_eng['body_drive_encoded'].isna().sum()}")
print(f"Missing after encoding - testing   : {testing_df_eng['body_drive_encoded'].isna().sum()}")
print()
print("Remaining columns:", training_df_eng.columns.tolist())


Missing after encoding - training  : 0
Missing after encoding - validation: 0
Missing after encoding - testing   : 0

Remaining columns: ['kilometres_driven', 'manufacturing_year', 'transmission_type', 'engine_cylinders', 'price', 'brand_bodytype_encoded', 'body_drive_encoded']


In [141]:
feature_engineering_2_explanations = """

In EDA I found that both body_type and drive_type individually had clear relationships with price.
So combining them would make it a good factor, for example a SUV_4WD is priced very differently from
a SUV_Front even though they are both SUVs. A 4WD SUV commands a significant premium over a front
wheel drive SUV.

The combination produced only 37 unique combinations in training which is very manageable and much
lower than the 220 combinations in brand_bodytype. This reduces the risk of overfitting significantly.

The same smoothed mean encoding approach was used as in C.1 with k = 10 as the smoothing factor.
Coupe_AWD and Convertible_Rear sit at the top while Sedan_Front and Hatchback_Front are at the bottom.

The original columns were later dropped to avoid redundancy in the model.

"""

In [142]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_2_explanations', value=feature_engineering_2_explanations)

### C.3 New Features considered and rejected



In [143]:
feature_engineering_rejection_explanations = """

During feature engineering I explored several ideas beyond brand_bodytype_encoded and body_drive_encoded.

vehicle_age (2026 - manufacturing_year):
Given that depreciation is influenced by age, this seems like a natural feature. The issue was that
it cant be assumed when the dataset was gathered or when a listing was made. A vehicle produced in 2000
could have only been purchased and driven starting in 2020, therefore an age estimation based on 2026
didnt seem right.

km_per_year (kilometres_driven / vehicle_age):
This was an interesting idea to measure how intensively a car was driven per year. But it directly
depends on vehicle_age which we rejected above for the same reasons. Without a reliable age
calculation this ratio is meaningless.

is_luxury (binary flag for exotic brands):
I tried defining luxury brands as those with a median price above $100,000 from training. This gave
6 brands: Lamborghini, Rolls-Royce, McLaren, Ferrari, Bentley and Aston. However only 18 records in
training were from these brands. A model cannot learn meaningful patterns from 18 examples. Also,
the brand_bodytype_encoded feature already assigns these exotic combinations very high encoded values
so is_luxury would just be redundant information. So it was dropped.

location / is_major_city:
Location directly affects car prices. Cars in Sydney and Melbourne tend to be priced higher than
regional areas. But as location column had 449 missing values in the test set and we cannot really
impute the city, this idea was dropped. Another take, if the pincode was available we could map it
accordingly and it could be a strong influence in predicting the price.

brand_bodytype_cylinders:
I also considered a three way combination of brand, body type and cylinders. While this would capture
even more specific pricing information, the number of combinations would be more and many
would have only 1-2 records and would mean overfitting.

"""

In [144]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_rejection_explanations', value=feature_engineering_rejection_explanations)

---
## D. Data Preparation for Modeling

In [145]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets

try:
  X_train = training_df_eng.copy()
  X_val = validation_df_eng.copy()
  X_test = testing_df_eng.copy()

  y_train = X_train.pop(target_name)
  y_val = X_val.pop(target_name)
  y_test = X_test.pop(target_name)
except Exception as e:
  print(e)

In [146]:
print("X_train columns:", X_train.columns.tolist())
print("X_train shape:", X_train.shape)
print()
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)
print()
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)
print("y_test shape:", y_test.shape)

X_train columns: ['kilometres_driven', 'manufacturing_year', 'transmission_type', 'engine_cylinders', 'brand_bodytype_encoded', 'body_drive_encoded']
X_train shape: (8755, 6)

X_val shape: (2608, 6)
X_test shape: (2038, 6)

y_train shape: (8755,)
y_val shape: (2608,)
y_test shape: (2038,)


### D.1 Data Transformation "transmission_type"


In [147]:
X_train = pd.get_dummies(X_train, columns=["transmission_type"], drop_first=True, dtype=int)
X_val = pd.get_dummies(X_val, columns=["transmission_type"], drop_first=True, dtype=int)
X_test = pd.get_dummies(X_test, columns=["transmission_type"], drop_first=True, dtype=int)

print("Columns after encoding:")
print(X_train.columns.tolist())
print()
print(X_train["transmission_type_Manual"].value_counts())
print()
print(X_train.dtypes)

Columns after encoding:
['kilometres_driven', 'manufacturing_year', 'engine_cylinders', 'brand_bodytype_encoded', 'body_drive_encoded', 'transmission_type_Manual']

transmission_type_Manual
0    7209
1    1546
Name: count, dtype: int64

kilometres_driven           float64
manufacturing_year            int64
engine_cylinders              int64
brand_bodytype_encoded      float64
body_drive_encoded          float64
transmission_type_Manual      int64
dtype: object


In [148]:
data_transformation_1_explanations = """

Transformation: One-hot encoding for transmission_type

transmission_type had 2 categories after cleaning; Automatic and Manual. Models cannot work with
string categories directly so they need to be converted to numbers.

I used one-hot encoding with drop_first=True which drops the Automatic category and creates a single
binary column called transmission_type_Manual. A value of 1 means Manual and 0 means Automatic.

"""

In [149]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_1_explanations', value=data_transformation_1_explanations)

### D.2 Data Transformation "Scaling all numeric features"

In [150]:
numeric_cols = [
    "kilometres_driven",
    "manufacturing_year",
    "engine_cylinders",
    "brand_bodytype_encoded",
    "body_drive_encoded"
]

# fit scaler on training only
scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])

# apply same scaler to val and test
X_val[numeric_cols] = scaler.transform(X_val[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print("After scaling - X_train stats:")
print(X_train[numeric_cols].describe().round(3))

After scaling - X_train stats:
       kilometres_driven  manufacturing_year  engine_cylinders  \
count           8755.000            8755.000          8755.000   
mean               0.000              -0.000             0.000   
std                1.000               1.000             1.000   
min               -2.024             -16.323            -4.146   
25%               -0.730              -0.558            -0.485   
50%               -0.159               0.343            -0.485   
75%                0.582               0.794            -0.485   
max                5.377               1.019             6.839   

       brand_bodytype_encoded  body_drive_encoded  
count                8755.000            8755.000  
mean                    0.000               0.000  
std                     1.000               1.000  
min                    -1.418              -1.196  
25%                    -0.597              -0.613  
50%                    -0.190               0.120  
75%       

In [151]:
data_transformation_2_explanations = """

Transformation: Standard scaling for numeric features

The numeric features in our dataset have very different scales:
kilometres_driven ranges from 1 to 533,849
manufacturing_year ranges from 1940 to 2023
engine_cylinders ranges from 2 to 12
brand_bodytype_encoded and body_drive_encoded are mean encoded prices

Without proper scaling, features with large ranges like kilometres_driven would dominate the model
because of their magnitude, not because they are more important. This could be problematic as we are
training on linear models.

So StandardScaler was used to transform each feature to obtain mean as 0 and standard deviation as 1
using the formula:
scaled = (value - mean) / std

The scaler was fitted on training data and the same mean and standard deviation values were applied
to validation and test data.

"""

In [152]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_2_explanations', value=data_transformation_2_explanations)

### D.3 Data Transformation "log transform target variable"


In [153]:
y_train_log = np.log(y_train)
y_val_log = np.log(y_val)
y_test_log = np.log(y_test)

print(f"y_train skewness before log: {y_train.skew():.3f}")
print(f"y_train skewness after log : {y_train_log.skew():.3f}")
print()
print(f"y_train mean before log: ${y_train.mean():,.2f}")
print(f"y_train mean after log : {y_train_log.mean():.3f}")

y_train skewness before log: 24.515
y_train skewness after log : 0.251

y_train mean before log: $24,734.24
y_train mean after log : 9.917


In [154]:
data_transformation_3_explanations = """

Transformation: Log transform on target variable (price)
The price was heavily right skewed. This means the vast majority of cars
are priced between $5k and $60k but a small number of exotic cars stretch all the way to $1.5M.

For linear regression it is required for the residulas to be normally distributed. When the target
variable is heavily skewed the errors will also be skewed, which violates this assumption and hurts
model performance.

So, I applied the log transform to bring down the skewness. It changed from 24.5 to just 0.27 which
is very close to normal. This will significantly improve the performance of the linear regression
models.

"""

In [155]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_3_explanations', value=data_transformation_3_explanations)

---
## E. Save Datasets

> Do not change this code

In [156]:
# DO NOT MODIFY THE CODE IN THIS CELL

try:
  X_train.to_csv(at.folder_path / 'X_train.csv', index=False)
  y_train.to_csv(at.folder_path / 'y_train.csv', index=False)

  X_val.to_csv(at.folder_path / 'X_val.csv', index=False)
  y_val.to_csv(at.folder_path / 'y_val.csv', index=False)

  X_test.to_csv(at.folder_path / 'X_test.csv', index=False)
  y_test.to_csv(at.folder_path / 'y_test.csv', index=False)
except Exception as e:
  print(e)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=fbbefce8-41ae-47c6-bc64-96decd566c0b' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>